# Final Conclusions

1. Temporal features were successfully engineered from pickup timestamps to capture travel patterns throughout the day and week. These included pickup hour, day of the week, weekend indicators, and New York City-specific rush hour indicators.

2. Rush hour periods were carefully defined based on typical NYC commuting patterns (Monday–Friday, 7:00–10:00 AM and 4:00–8:00 PM) to better represent traffic congestion effects on trip duration and fare.

3. Spatial information was preserved by retaining pickup and dropoff location identifiers (`PULocationID` and `DOLocationID`), which provide critical insights into route geography and travel behavior.

4. A route-level feature (`Location_Pair`) was created by combining pickup and dropoff zones, enabling machine learning models to learn common route-specific patterns associated with trip duration and fare estimation.

5. Passenger count was retained as a valuable pre-trip feature because it is available before ride initiation and may influence trip characteristics.

6. Features with limited predictive value or poor generalizability were identified and removed. Specifically:
   - `pickup_month` was excluded because the dataset contained only January 2026 records, resulting in zero variance.
   - `pickup_day` was excluded to avoid learning date-specific patterns that may not generalize to future data.

7. Leakage-prone features unavailable at prediction time were removed to ensure realistic model performance. In particular, `tpep_dropoff_datetime` was excluded because it is only known after trip completion.

8. The feature engineering process was designed to align with the project's real-world objective of predicting trip duration and expected fare before ride initiation, ensuring that only information reasonably available at booking time is utilized.

9. Target variables (`trip_duration` and `fare_amount`) were retained within the final dataset for subsequent model development but will be excluded from input features during training.

10. The resulting dataset represents a clean, production-oriented, machine learning-ready dataset that balances predictive power, interpretability, and prevention of data leakage.

---

## Final Dataset Overview

### Predictive Features

- `trip_distance`
- `passenger_count`
- `PULocationID`
- `DOLocationID`
- `Location_Pair`
- `pickup_hour`
- `pickup_dayofweek`
- `is_weekend`
- `is_rush_hour`

### Target Variables

- `trip_duration`
- `fare_amount`

---

## Outcome

The dataset has been successfully transformed into a robust feature-engineered dataset suitable for developing high-quality machine learning models capable of estimating trip duration and fare prior to ride initiation.

---

## Next Steps

The next notebook will focus on:

- Exploratory Data Analysis (EDA)
- Understanding feature distributions and relationships
- Investigating patterns affecting trip duration and fare
- Identifying insights to guide model selection and feature importance analysis
- Preparing the dataset for machine learning model development

---
---

# Feature Engineering

## Objective

The objective of this notebook is to create meaningful features from the cleaned NYC Yellow Taxi dataset that can improve the predictive performance of machine learning models developed for:

- Trip Duration Prediction
- Expected Fare Prediction

Feature engineering focuses on extracting useful temporal and spatial information available before ride initiation while ensuring that no data leakage is introduced.

The following tasks will be performed:

- Extract temporal features from pickup timestamps.
- Create traffic-related indicators.
- Generate route-based spatial features.
- Validate engineered features.
- Remove leakage-prone variables.
- Save the final feature-engineered dataset for modeling.

# Import Required Libraries

The following libraries are used for data manipulation, visualization, and feature engineering.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# Load Cleaned Dataset

The cleaned dataset generated during the data cleaning phase is loaded for feature engineering.

In [3]:
df = pd.read_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/data/cleaned_taxi_data.parquet"
)

print(df.shape)

df.head()

(3499962, 10)


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,fare_amount,cbd_congestion_fee,trip_duration
0,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,239,238,7.2,0.00,5.550000
1,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,142,209,38.7,0.75,42.800000
2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,144,137,14.2,0.75,13.600000
3,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,142,50,11.4,0.75,10.633333
4,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,161,45,37.3,0.75,37.733333


# Dataset Inspection

Before creating new features, the structure of the cleaned dataset is reviewed to ensure consistency and data integrity.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3499962 entries, 0 to 3499961
Data columns (total 10 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   RatecodeID             float64       
 5   PULocationID           int32         
 6   DOLocationID           int32         
 7   fare_amount            float64       
 8   cbd_congestion_fee     float64       
 9   trip_duration          float64       
dtypes: datetime64[us](2), float64(6), int32(2)
memory usage: 240.3 MB


In [5]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
tpep_pickup_datetime,3499962,2026-01-17 00:51:03.487804,2025-12-31 23:57:29,2026-01-09 17:35:59,2026-01-16 20:57:27.500000,2026-01-24 03:41:31.500000,2026-02-01 00:45:01,NaN
tpep_dropoff_datetime,3499962,2026-01-17 01:08:10.202258,2025-12-31 23:57:32,2026-01-09 17:54:11.250000,2026-01-16 21:13:05.500000,2026-01-24 03:54:22,2026-02-01 01:03:08,NaN
passenger_count,3499962.0,1.186869,1.0,1.0,1.0,1.0,6.0,0.573038
trip_distance,3499962.0,3.482794,0.01,1.09,1.9,3.89,95.1,4.195791
RatecodeID,2505383.0,5.359114,1.0,1.0,1.0,1.0,99.0,20.018976
PULocationID,3499962.0,161.495252,1.0,114.0,161.0,233.0,265.0,67.022332
DOLocationID,3499962.0,161.05483,1.0,107.0,162.0,234.0,265.0,71.022493
fare_amount,3499962.0,21.020974,0.01,10.0,15.6,26.1,200.0,16.76218
cbd_congestion_fee,3499962.0,0.531944,0.0,0.0,0.75,0.75,0.75,0.340578
trip_duration,3499962.0,17.111908,0.016667,8.333333,13.533333,21.333333,179.966667,13.753058


## Pickup Hour

Trip duration and fare patterns vary throughout the day due to traffic conditions and travel demand.

Extracting the pickup hour enables the model to capture intraday variations.

In [6]:
df["pickup_hour"] = (
    df["tpep_pickup_datetime"]
    .dt.hour
)
df["pickup_hour"].value_counts().sort_index()

pickup_hour
0     108025
1      74927
2      51970
3      37128
4      27452
5      31641
6      61424
7     109430
8     142820
9     150750
10    155016
11    164834
12    179841
13    187466
14    198745
15    212322
16    202384
17    219963
18    228614
19    208154
20    205665
21    205426
22    188454
23    147511
Name: count, dtype: int64

## Day of Week

Travel behavior differs across weekdays and weekends.

This feature captures weekly travel patterns.

In [7]:
df["pickup_dayofweek"] = (
    df["tpep_pickup_datetime"]
    .dt.dayofweek
)
df["pickup_dayofweek"].value_counts().sort_index()

pickup_dayofweek
0    352410
1    450450
2    471462
3    611461
4    607385
5    647534
6    359260
Name: count, dtype: int64

## Month

Month is extracted to support scalability if future datasets covering multiple months are incorporated.

In [8]:
df["pickup_month"] = (
    df["tpep_pickup_datetime"]
    .dt.month
)

df["pickup_month"].value_counts()

pickup_month
1     3499956
12          5
2           1
Name: count, dtype: int64

## Day of Month

Daily travel demand patterns may vary throughout the month.

In [9]:
df["pickup_day"] = (
    df["tpep_pickup_datetime"]
    .dt.day
)

df["pickup_day"].describe()

count    3.499962e+06
mean     1.642972e+01
std      8.858874e+00
min      1.000000e+00
25%      9.000000e+00
50%      1.600000e+01
75%      2.400000e+01
max      3.100000e+01
Name: pickup_day, dtype: float64

## Weekend Indicator

Travel patterns during weekends often differ from weekdays.

This binary indicator identifies trips occurring on Saturdays and Sundays.

In [10]:
df["is_weekend"] = np.where(
    df["pickup_dayofweek"].isin([5, 6]),
    1,
    0
)
df["is_weekend"].value_counts()

is_weekend
0    2493168
1    1006794
Name: count, dtype: int64

## Rush Hour Indicator

Traffic congestion is typically higher during commuting periods.

Rush hour periods are defined as:

- Morning: 7 AM – 10 AM
- Evening: 4 PM – 7 PM

In [11]:
df["is_rush_hour"] = np.where(
    (df["pickup_hour"].between(7, 10)) | (df["pickup_hour"].between(16, 19)),
    1,
    0
)

df["is_rush_hour"].value_counts()

is_rush_hour
0    2082831
1    1417131
Name: count, dtype: int64

## Spatial Feature Engineering
## Route Feature (Location Pair)


The combination of pickup and dropoff locations represents the taxi route.

This feature enables the model to learn route-specific travel patterns and fare characteristics.


In [12]:
df["Location_Pair"] = (
    df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
)
df["Location_Pair"].sample(10)

618508       76_61
1805892    186_263
513846     114_255
2908543    161_262
1301575      43_90
1504257     233_90
3434164    246_143
387566     236_162
1394949    231_211
2703652    163_231
Name: Location_Pair, dtype: object

## Geographic Features

Pickup and dropoff location identifiers provide valuable spatial information that influences both travel time and fare amount.

These variables are retained for modeling purposes.

In [13]:
df[
    [
        "PULocationID",
        "DOLocationID"
    ]
].describe()

,PULocationID,DOLocationID
count,3.499962e+06,3.499962e+06
mean,1.614953e+02,1.610548e+02
std,6.702233e+01,7.102249e+01
min,1.000000e+00,1.000000e+00
25%,1.140000e+02,1.070000e+02
50%,1.610000e+02,1.620000e+02
75%,2.330000e+02,2.340000e+02
max,2.650000e+02,2.650000e+02


In [14]:
print(
    "Unique Pickup Zones:",
    df["PULocationID"].nunique()
)

print(
    "Unique Dropoff Zones:",
    df["DOLocationID"].nunique()
)

Unique Pickup Zones: 262
Unique Dropoff Zones: 260


## Passenger Count

Passenger count is retained as a predictive feature because it is available before trip initiation and may influence trip characteristics.

In [15]:
df["passenger_count"].value_counts()

passenger_count
1.0    3052767
2.0     318635
3.0      69048
4.0      45593
5.0       9070
6.0       4849
Name: count, dtype: int64

# Feature Validation

The engineered features are reviewed to ensure successful creation and consistency.

In [16]:
engineered_features = [

    "pickup_hour",

    "pickup_dayofweek",

    "pickup_month",

    "pickup_day",

    "is_weekend",

    "is_rush_hour",

    "Location_Pair"

]

df[
    engineered_features
].sample(10)

,pickup_hour,pickup_dayofweek,pickup_month,pickup_day,is_weekend,is_rush_hour,Location_Pair
770061,20,5,1,10,1,0,249_13
2441131,11,5,1,31,1,0,141_263
2597665,23,5,1,3,1,0,143_163
340965,21,0,1,5,0,0,90_107
996333,16,1,1,13,0,1,162_140
257642,18,6,1,4,1,1,151_238
421905,21,1,1,6,0,0,237_236
336102,19,0,1,5,0,1,132_230
2262560,12,3,1,29,0,0,161_237
1299427,17,4,1,16,0,1,148_237


# Leakage Feature Removal

Dropoff timestamps were previously used to derive trip duration.

Since dropoff information is unavailable before ride completion, these features must be removed before model development to prevent data leakage.

In [17]:
df.drop(
    columns=[
        "tpep_dropoff_datetime",
    ],
    inplace=True
)


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3499962 entries, 0 to 3499961
Data columns (total 16 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   passenger_count       float64       
 2   trip_distance         float64       
 3   RatecodeID            float64       
 4   PULocationID          int32         
 5   DOLocationID          int32         
 6   fare_amount           float64       
 7   cbd_congestion_fee    float64       
 8   trip_duration         float64       
 9   pickup_hour           int32         
 10  pickup_dayofweek      int32         
 11  pickup_month          int32         
 12  pickup_day            int32         
 13  is_weekend            int64         
 14  is_rush_hour          int64         
 15  Location_Pair         object        
dtypes: datetime64[us](1), float64(6), int32(6), int64(2), object(1)
memory usage: 347.1+ MB


In [19]:
df.drop(columns=["pickup_month"], inplace=True)
df.drop(columns=["cbd_congestion_fee"], inplace=True)
df.drop(columns=["RatecodeID"], inplace=True)
df.drop(columns=["fare_amount"], inplace=True)


# Final Dataset Validation

Review the final structure of the feature-engineered dataset before saving.

In [20]:
print(
    "Final Dataset Shape:",
    df.shape
)

df.head()

Final Dataset Shape: (3499962, 12)


,tpep_pickup_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,trip_duration,pickup_hour,pickup_dayofweek,pickup_day,is_weekend,is_rush_hour,Location_Pair
0,2026-01-01 00:54:04,1.0,0.97,239,238,5.550000,0,3,1,0,0,239_238
1,2026-01-01 00:15:22,4.0,5.58,142,209,42.800000,0,3,1,0,0,142_209
2,2026-01-01 00:47:11,2.0,2.33,144,137,13.600000,0,3,1,0,0,144_137
3,2026-01-01 00:17:54,1.0,1.30,142,50,10.633333,0,3,1,0,0,142_50
4,2026-01-01 00:34:14,1.0,5.34,161,45,37.733333,0,3,1,0,0,161_45


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3499962 entries, 0 to 3499961
Data columns (total 12 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   passenger_count       float64       
 2   trip_distance         float64       
 3   PULocationID          int32         
 4   DOLocationID          int32         
 5   trip_duration         float64       
 6   pickup_hour           int32         
 7   pickup_dayofweek      int32         
 8   pickup_day            int32         
 9   is_weekend            int64         
 10  is_rush_hour          int64         
 11  Location_Pair         object        
dtypes: datetime64[us](1), float64(3), int32(5), int64(2), object(1)
memory usage: 253.7+ MB


# Save Feature-Engineered Dataset

The processed dataset is saved for use in exploratory data analysis and model development.

In [22]:
df.to_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/data/feature_engineered_taxi_data.parquet",
    index=False
)

In [5]:
saved_df = pd.read_parquet(
    "/home/ed/Desktop/NYC_Taxi_Demand_Forecasting/USECASE2_Trip_Fare_Forecasting/data/feature_engineered_taxi_data.parquet"
)

print(saved_df.shape)

NameError: name 'pd' is not defined

In [4]:
type(saved_df['trip_duration'])

NameError: name 'saved_df' is not defined